# Séance 4 — Du score à la preuve

Suite du notebook `B1_S04_supervised_ml_baseline.ipynb`.

**Question : un score parfait suffit-il pour faire confiance à un modèle ?**

Avant d'exécuter la suite, proposer trois vérifications du premier résultat. Nous allons examiner la cible et les classes, puis tester une autre question de prédiction sur le même jeu de données. Les scores des deux tâches ne sont pas directement comparables.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.model_selection import TimeSeriesSplit

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/snapshots/iot_FR_today_5-y_chatgpt_iphone_meteo.csv").exists())
raw = pd.read_csv(ROOT / "data/snapshots/iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
raw = raw.sort_values("date").reset_index(drop=True)
signals = ["chatgpt", "iphone", "meteo"]
raw[signals] = raw[signals].apply(pd.to_numeric, errors="coerce")
OUT = ROOT / "outputs/bloc1/s04_diagnostic"
OUT.mkdir(parents=True, exist_ok=True)

## 1. Diagnostiquer le premier résultat

Reprenons la construction de la cible initiale à partir du snapshot. Si le premier notebook utilisait un fichier nettoyé différent, les chiffres peuvent varier.

- Le seuil médian est calculé sur toute la série : la définition de la cible utilise donc aussi le futur.
- Une séparation chronologique peut laisser une seule classe en validation.
- Une accuracy de 100 % peut alors être obtenue sans exploiter les variables explicatives.

Le dernier label est inconnu : comparer une valeur manquante à un seuil le transforme artificiellement en zéro. Il faut l'exclure explicitement avant l'évaluation.

In [ ]:
audit = raw.copy()
for c in signals:
    audit[f"{c}_lag1"] = audit[c].shift(1)
    audit[f"{c}_roll4"] = audit[c].rolling(4).mean()
audit["target"] = (audit["chatgpt"].shift(-1) > audit["chatgpt"].median()).astype(int)
audit = audit.dropna().iloc[:-1].reset_index(drop=True)
k = int(len(audit) * 0.8)
display(pd.DataFrame({"train": audit.iloc[:k].target.value_counts(),
                      "validation": audit.iloc[k:].target.value_counts()}).fillna(0).astype(int))
print("Accuracy si l'on prédit toujours 1 :", (audit.iloc[k:].target == 1).mean())

## 2. Une question définie sans seuil calculé dans le futur

À la fin de la semaine t, peut-on prédire si l'intérêt pour ChatGPT sera **strictement supérieur** en semaine t+1 ? Une égalité est classée 0.

Les variables utilisent uniquement les semaines connues à t. Les moyennes mobiles incluent t, ce qui est compatible avec une prédiction faite en fin de semaine. Cette cible mesure une hausse du signal Google Trends, pas une hausse de ventes ni un effet causal. Le snapshot reste un exemple pédagogique rétrospectif ; il ne garantit pas la disponibilité historique exacte des valeurs normalisées.

In [ ]:
X = pd.DataFrame(index=raw.index)
for c in signals:
    X[c] = raw[c]
    X[f"{c}_lag1"] = raw[c].shift(1)
    X[f"{c}_change1"] = raw[c].diff()
    X[f"{c}_roll4"] = raw[c].rolling(4, min_periods=4).mean()
X["month"] = raw.date.dt.month
next_value = raw.chatgpt.shift(-1)
known = raw.chatgpt.notna() & next_value.notna() & X.notna().all(axis=1)
X = X.loc[known].copy()
y = (next_value.loc[known] > raw.loc[known, "chatgpt"]).astype(int)
dates = raw.loc[known, "date"]
label_dates = raw.date.shift(-1).loc[known]
cut = int(len(X) * 0.8)
# Le label de chaque ligne d'entraînement doit être connu avant la validation.
train = (np.arange(len(X)) < cut) & (label_dates < dates.iloc[cut]).to_numpy()
valid = np.arange(len(X)) >= cut
X_train, X_valid = X.loc[train], X.loc[valid]
y_train, y_valid = y.loc[train], y.loc[valid]
display(pd.DataFrame({"train": y_train.value_counts(), "validation": y_valid.value_counts()}).fillna(0).astype(int))
assert y_train.nunique() == y_valid.nunique() == 2, "Vérifier les classes avant d'interpréter les scores."
print("Train :", dates.loc[train].min(), "→", dates.loc[train].max())
print("Validation :", dates.loc[valid].min(), "→", dates.loc[valid].max())

## 3. Comparer à des références simples

Le modèle apprend le remplacement des valeurs manquantes et la standardisation uniquement sur l'entraînement. Deux références : prédire la classe majoritaire du train, ou prolonger le sens du dernier mouvement. La balanced accuracy donne le même poids aux deux classes ; le F1 porte ici sur la classe « hausse ». La ROC AUC évalue le classement des probabilités.

Le bloc final de validation est réservé à cette comparaison. Ne pas régler les paramètres en regardant ses résultats.

In [ ]:
def new_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))

def score(name, truth, pred, proba=None):
    return {"model": name, "accuracy": accuracy_score(truth, pred),
            "balanced_accuracy": balanced_accuracy_score(truth, pred),
            "f1_hausse": f1_score(truth, pred, zero_division=0),
            "roc_auc": roc_auc_score(truth, proba) if proba is not None and truth.nunique() == 2 else np.nan}

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = new_model().fit(X_train, y_train)
persistence = (X_valid.chatgpt_change1 > 0).astype(int)
pred = model.predict(X_valid)
metrics = pd.DataFrame([
    score("majorité train", y_valid, baseline.predict(X_valid), baseline.predict_proba(X_valid)[:, 1]),
    score("dernier mouvement", y_valid, persistence),
    score("régression logistique", y_valid, pred, model.predict_proba(X_valid)[:, 1]),
])
display(metrics.round(3))
metrics.to_csv(OUT / "metrics.csv", index=False)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_valid, pred, labels=[0, 1],
    display_labels=["stable / baisse", "hausse"], ax=ax, colorbar=False)
ax.set_title("Régression logistique — validation chronologique")
plt.tight_layout()
plt.show()

## 4. La performance est-elle stable dans le temps ?

Effectuer trois évaluations à fenêtre d'entraînement croissante **à l'intérieur du train**. Une ligne d'écart empêche le label de la dernière semaine d'entraînement d'utiliser la première semaine de validation. Si une période ne contient qu'une classe, elle est signalée et non transformée en score flatteur.

In [ ]:
folds = []
for fold, (a, b) in enumerate(TimeSeriesSplit(n_splits=3, gap=1).split(X_train), 1):
    ya, yb = y_train.iloc[a], y_train.iloc[b]
    if ya.nunique() < 2 or yb.nunique() < 2:
        print(f"Période {fold} : une seule classe, comparaison non interprétable.")
        continue
    for name, clf in [("majorité train", DummyClassifier(strategy="most_frequent")),
                      ("régression logistique", new_model())]:
        clf.fit(X_train.iloc[a], ya)
        result = score(name, yb, clf.predict(X_train.iloc[b]), clf.predict_proba(X_train.iloc[b])[:, 1])
        folds.append({"fold": fold, **result})
cv_metrics = pd.DataFrame(folds)
display(cv_metrics.round(3))
cv_metrics.to_csv(OUT / "temporal_validation.csv", index=False)

## À discuter

1. Le modèle bat-il les deux références sur la validation finale ? Sur quelles métriques ?
2. Les évaluations successives racontent-elles la même histoire ?
3. Quel coût donner aux fausses alertes et aux hausses manquées ?
4. Quelles données supplémentaires faudrait-il avant d'utiliser cette prédiction ?

**Conclusion :** un notebook qui s'exécute n'est pas nécessairement une expérience valide. Le choix de la cible, les classes présentes et la comparaison aux références comptent davantage qu'un score spectaculaire. Une performance décevante est un résultat à expliquer, pas à masquer.